# Experiment 1 — CSSNet baseline

Open in **VS Code** (Jupyter + Google Colaboratory). Select Kernel → **Colab** → **T4 GPU**.

On My Drive: `cv_dataset.zip` only. This notebook clones the code from GitHub. Checkpoints go to `MyDrive/cssnet_runs/exp1`.


In [ ]:
import torch
assert torch.cuda.is_available(), "Select Kernel → Colab → T4 GPU"
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_RUNS = "/content/drive/MyDrive/cssnet_runs/exp1"
GIT_URL = "https://github.com/Areesha-008/Abdominal-CT-Segmentation.git"

_mydrive = Path("/content/drive/MyDrive")
_candidates = [
    _mydrive / "cv_dataset.zip",
    _mydrive / "cv_dataset.tar.gz",
    _mydrive / "cv_dataset.tgz",
    _mydrive / "cv_dataset.rar",
    _mydrive / "cv_dataset",
]
DRIVE_DATA = next((p for p in _candidates if p.exists()), None)
assert DRIVE_DATA is not None, "Upload cv_dataset.zip to My Drive (must contain train/ and test/)"
print("data archive:", DRIVE_DATA)


In [ ]:
from pathlib import Path
import shutil

CODE = Path("/content/Abdominal-CT-Segmentation")
DATA = Path("/content/Synapse")
UNPACK = Path("/content/_cv_dataset_unpack")

if (CODE / "train.py").exists():
    %cd {CODE}
    !git fetch --depth 1 origin
    !git reset --hard origin/main
else:
    !git clone --depth 1 {GIT_URL} {CODE}
    %cd {CODE}

def find_split(root: Path, name: str, pattern: str):
    hits = [p.parent for p in root.rglob(pattern) if p.parent.name == name]
    return hits[0] if hits else None

if not (DATA / "train").is_dir():
    src = Path(DRIVE_DATA)
    if UNPACK.exists():
        shutil.rmtree(UNPACK)
    UNPACK.mkdir(parents=True)

    if src.is_file() and src.suffix == ".zip":
        !unzip -q -o "{src}" -d "{UNPACK}"
    elif src.is_file() and (src.suffixes[-2:] == [".tar", ".gz"] or src.suffix == ".tgz"):
        !tar -xzf "{src}" -C "{UNPACK}"
    elif src.is_file() and src.suffix == ".rar":
        !apt-get -qq install -y unrar
        !unrar x -o+ "{src}" "{UNPACK}/"
    elif src.is_dir():
        !cp -a "{src}" "{UNPACK}/cv_dataset"
    else:
        raise FileNotFoundError(src)

    train_src = find_split(UNPACK, "train", "*.npz")
    test_src = find_split(UNPACK, "test", "*.h5")
    assert train_src and test_src, f"archive must contain train/*.npz and test/*.h5; top level: {list(UNPACK.iterdir())[:20]}"
    DATA.mkdir(parents=True, exist_ok=True)
    shutil.move(str(train_src), str(DATA / "train"))
    shutil.move(str(test_src), str(DATA / "test"))
    shutil.rmtree(UNPACK, ignore_errors=True)

print("train slices", len(list((DATA / "train").glob("*.npz"))))
print("test volumes", len(list((DATA / "test").glob("*.h5"))))


In [ ]:
%pip install -q scipy h5py opencv-python-headless timm ml-collections medpy SimpleITK pyyaml


In [ ]:
from pathlib import Path
resume = "--resume" if Path(DRIVE_RUNS, "last.pth").exists() else ""
print("resume" if resume else "fresh run")
!python train.py --config config.yaml --out "{DRIVE_RUNS}" --data_root "{DATA}" {resume}


In [ ]:
!python evaluate.py --config config.yaml --run "{DRIVE_RUNS}" --data_root "{DATA}"


In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

r = json.load(open(Path(DRIVE_RUNS) / "results.json"))
print(f"{'organ':14s} {'Dice':>8s} {'IoU':>8s}")
for o, v in r["per_organ"].items():
    print(f"{o:14s} {v['dice']:8.4f} {v['iou']:8.4f}")
print(f"{'MEAN':14s} {r['mean_dice']:8.4f} {r['mean_iou']:8.4f}")

for p in sorted((Path(DRIVE_RUNS) / "overlays").glob("*.png")):
    display(Image(filename=str(p)))
